# Gadda's correspondence → tables ready for Linked Open Data

This notebook turns the two Excel files on the letters into clean tables, following the
**LetterSampo** model (*Letter*, *Actor*, *Place*, *Tie*), ready for the RDF conversion.

**The idea**
- `dati_mittenti_Gadda.xlsx` is the **backbone**: every correspondent in it becomes an *actor*,
  with a *tie* (their correspondence with Gadda). Its numbers count **only the letters Gadda
  received** (the Bonsanti and Liberati inventories).
- `lettere.xlsx` adds the **individual letters**, each attached to its correspondence.
- Identifiers come **only from the corrected OpenRefine exports** (`lod/openrefine/tables`) and
  from your checked list `viaf_to_check.csv`.
- Every person, place and correspondence gets its **own OpenGadda id** (p0001, pl0001, t0001…).
  External ids (VIAF, Wikidata, GeoNames) are attached to it as links.
- Nothing is invented: when something is missing it stays missing, and it's listed in `report.md`.

**What it writes** (in `lod/letters/`):

| file | content |
|---|---|
| `ids.csv` | the OpenGadda ids, kept between runs |
| `people.csv` | every actor, with ids and statistics |
| `ties.csv` | one row per correspondence with Gadda |
| `letters.csv` | one row per letter |
| `places.csv` | every place, with city / address / GeoNames |
| `sources.csv` | editions and archives the information comes from |
| `report.md` | what is missing or needs a decision |

The Excel files are **only read**, never changed. Run the cells from top to bottom
(*Run All*); after you correct the Excel, just run everything again.

## 1. Settings

Where the files are, and a few fixed choices:
- `VIAF_CHECKED`: your checked VIAF list is used (you went through it on 2026-11-25).
- `ORG_CATEGORIES`: correspondents in this category are always organisations. "Editori, riviste
  e quotidiani" isn't here, because it also includes people (Garzanti Livio, Einaudi Giulio).
- `SET_ASIDE`: names you asked to leave out of the problem lists for now.

In [1]:
import re
import csv
import json
import collections
from pathlib import Path

import pandas as pd

HERE = Path('/Users/martinapensalfini/Documents/opengadda/lod/letters')   # where the tables are written
LOD = HERE.parent
ONEDRIVE = Path('/Users/martinapensalfini/Library/CloudStorage/OneDrive-AlmaMaterStudiorumUniversitàdiBologna/dataset')
LETTERS_XLSX = ONEDRIVE / 'lettere.xlsx'
SENDERS_XLSX = ONEDRIVE / 'dati_mittenti_Gadda.xlsx'
TABLES = LOD / 'openrefine' / 'tables'          # the corrected OpenRefine exports
IDS_CSV = HERE / 'ids.csv'
VIAF_CHECK_CSV = HERE / 'viaf_to_check.csv'
VIAF_CHECKED = True                             # your checked list is used

GADDA = 'Gadda Carlo Emilio'
ITEMS_JSON = LOD.parent / 'code_updated' / 'items.json'   # only to link editions to their page on the site
SITE_RECORD = 'https://angstigone.github.io/opengadda/code_updated/detail.html?id='
ORG_CATEGORIES = {'Istituzioni, associazioni ed enti pubblici'}
SET_ASIDE = {'pasquotti famiglia'}              # no letter count in the senders file: ignored for now

## 2. Corrections agreed so far

Fixes applied while reading the Excel, so the Excel itself stays as you keep it:
- **dates**: Let1511 was "2959" (→ 1959); Let1672 said "monda" (→ undated until you know the date);
- **places with two GeoNames ids**: keep the **city** (populated place), not the administrative area
  (checked on geonames.org).

In [2]:
DATE_FIX = {
    'Let1511': '1959-07-24',   # "2959-07-24" in the Excel
    'Let1672': None,           # "monda" in the Excel: date unknown for now
}
CITY_ID = {
    frozenset({'6542283', '3173435'}): '3173435',  # Milan
    frozenset({'2968815', '2988507'}): '2988507',  # Paris
    frozenset({'6453507', '2983013'}): '2983013',  # Romans
    frozenset({'6541857', '3176885'}): '3176885',  # Foggia
    frozenset({'7287487', '2658112'}): '2658112',  # Visp
    frozenset({'6541860', '2525764'}): '2525764',  # Agrigento
}

## 3. Small helpers

- `clean`: tidies a cell (no extra spaces; empty cells become `''`).
- `key`: the form used to **recognise the same name** in different files: lower case, no commas
  ("Contini, Gianfranco" and "Contini Gianfranco" become the same key).
- `split_names`: a cell with several people separated by `;` becomes a list; `?` means unknown.

In [3]:
def clean(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ''
    return re.sub(r'\s+', ' ', str(v)).strip()


def key(name):
    return re.sub(r'\s+', ' ', str(name).replace(',', '')).strip().lower()


def split_names(v):
    return [n for n in (clean(x) for x in clean(v).split(';')) if n and n != '?']


split_names('Gobbi Ambrogio; Marchetti Domenico; ?')   # example

['Gobbi Ambrogio', 'Marchetti Domenico']

## 4. OpenGadda ids that never change

Every person gets `p` + a number (Gadda is `p0001`), every place `pl` + a number. They're saved in
`ids.csv`: when you run the notebook again, **a name already there keeps its number**, and only new
names get the next free number. So a published id always points to the same person.

In [4]:
PREFIX = {'person': 'p', 'place': 'pl'}
ids = {}          # (kind, key) → id
if IDS_CSV.exists():
    with open(IDS_CSV, encoding='utf-8-sig') as f:
        for r in csv.DictReader(f):
            ids[(r['kind'], r['key'])] = r['id']
labels = {}


def og_id(kind, k, label):
    labels[(kind, k)] = label
    if (kind, k) not in ids:
        used = [int(re.sub(r'\D', '', v)) for (kd, _), v in ids.items() if kd == kind]
        ids[(kind, k)] = f'{PREFIX[kind]}{(max(used) + 1 if used else 1):04d}'
    return ids[(kind, k)]


print(len(ids), 'ids already assigned')

1389 ids already assigned


## 5. The corrected authorities (OpenRefine exports + your VIAF check)

**People.** `names-in-items-csv.csv` has one row per name in each record, with its VIAF and its type
(person / organisation). Note: `name_plain` now holds VIAF's form of the name ("Fabrizio Clerici"),
so names are recognised by **`name_as_written`** ("Clerici Fabrizio"), which is how the Excel
writes them. `people-csv.csv` adds Wikidata and birth/death years.

**Your checked list.** `viaf_to_check.csv` overrides everything else for the people in it: the id
left in `your_viaf` or `old_viaf` is used, an empty cell means "no VIAF". Excel had turned some long
ids into numbers like `1,29152E+20`, so those are recovered from the `viaf_link` column.

In [5]:
VIAF_COL = 'VIAF cluster ID'
names = pd.read_csv(TABLES / 'names-in-items-csv.csv', dtype=str).fillna('')
by_item = {}                                    # (letter id, name) → that row
viaf_by_name = collections.defaultdict(set)     # name → VIAF ids found for it
type_by_name = {}                               # name → person / organization
for r in names.to_dict('records'):
    for n in {r['name_as_written'], r['name_plain']}:
        k = key(n)
        if r['category'] == 'letter':
            by_item.setdefault((r['item_id'], k), r)
        if r[VIAF_COL]:
            viaf_by_name[k].add(r[VIAF_COL])
        if r['type'] and k not in type_by_name:
            type_by_name[k] = r['type']

people_tab = pd.read_csv(TABLES / 'people-csv.csv', dtype=str).fillna('')
extra = {}                                      # name → Wikidata, birth, death
for r in people_tab.to_dict('records'):
    k = key(r['name'])
    extra[k] = r
    if r[VIAF_COL]:
        viaf_by_name[k].add(r[VIAF_COL])

checked = {}                                    # your decisions: name → VIAF ('' = none)
if VIAF_CHECKED and VIAF_CHECK_CSV.exists():
    text = VIAF_CHECK_CSV.read_text(encoding='utf-8-sig')
    first = text.split('\n', 1)[0]
    sep = ';' if first.count(';') > first.count(',') else ','
    for r in csv.DictReader(text.splitlines(), delimiter=sep):
        def viaf(col):
            v = clean(r.get(col))
            if re.search(r'[eE]\+', v):          # mangled by Excel → take it from the link
                m = re.search(r'viaf/(\d+)', r.get('viaf_link', ''))
                v = m[1] if m else ''
            return v
        checked[key(r['correspondent'])] = viaf('your_viaf') or viaf('old_viaf')

print(len(viaf_by_name), 'names with a VIAF in the exports;', len(checked), 'decisions from your check')

3726 names with a VIAF in the exports; 131 decisions from your check


**Places.** `places-csv.csv` gives, for each place as written, its city, address and GeoNames id.
When OpenRefine split a value over several rows (a second GeoNames id on the next row, with the
other cells empty), those rows are put back together here.

`place_of` then turns a place as written into: its city (with its GeoNames id), and, if there's an
address, a separate small place **within** that city (e.g. "Roma (Blumenstihl 19)" → *Rome* →
*Rome, Blumenstihl 19*). Each gets its own OpenGadda id.

In [6]:
place_rows = pd.read_csv(TABLES / 'places-csv.csv', dtype=str).fillna('')
place_auth = {}
cur = None
for r in place_rows.to_dict('records'):
    if r['field']:
        cur = {'value': clean(r['value']), 'city': clean(r['city']),
               'ids': [r['GeoNames ID']] if r['GeoNames ID'] else [], 'address': clean(r['address'])}
        if r['field'] in ('luogo_di_invio', 'luogo_destinatario'):
            place_auth.setdefault(cur['value'], cur)
    elif cur is not None and r['GeoNames ID']:   # continuation row: another id for the same place
        cur['ids'].append(r['GeoNames ID'])


def place_of(raw):
    raw = clean(raw)
    if not raw:
        return {'raw': '', 'id': '', 'city': '', 'address': '', 'geonames': '', 'city_id': ''}
    a = place_auth.get(raw)
    city = a['city'] if a else ''
    gid = (CITY_ID.get(frozenset(a['ids']), a['ids'][0]) if a and a['ids'] else '')
    address = a['address'] if a else ''
    city_key = gid or key(city or raw)
    city_id = og_id('place', city_key, city or raw)
    pid = og_id('place', f'{city_key}|{key(address)}', f'{city}, {address}') if address else city_id
    return {'raw': raw, 'id': pid, 'city': city, 'address': address, 'geonames': gid, 'city_id': city_id}


place_of('Roma (Blumenstihl 19)')   # example

{'raw': 'Roma (Blumenstihl 19)',
 'id': 'pl0148',
 'city': 'Rome',
 'address': 'Blumenstihl 19',
 'geonames': '3169070',
 'city_id': 'pl0008'}

## 6. Actors and their correspondences (the senders file)

`actor` creates a person or organisation the first time a name appears (with its OpenGadda id,
VIAF, Wikidata, dates and type) and returns the same record every following time.

Then every row of `dati_mittenti_Gadda.xlsx` becomes a **tie**: the correspondence between that
person and Gadda, with the number of letters **Gadda received** from them in each inventory and
their date ranges. Five people are on two rows (Gadda Enrico, Martini Quinto…): their rows are
merged into one tie, adding up the numbers.

In [7]:
people = {}


def actor(name, category=''):
    k = key(name)
    p = people.get(k)
    if p is None:
        viafs = sorted(viaf_by_name.get(k, set()))
        if k in checked:                         # your decision wins
            viafs = [checked[k]] if checked[k] else []
        x = extra.get(k, {})
        typ = 'organization' if category in ORG_CATEGORIES else (type_by_name.get(k) or 'person')
        p = people[k] = {
            'id': og_id('person', k, name), 'name': name, 'type': typ,
            'viaf': ' | '.join(viafs), 'wikidata': x.get('wikidata_id', ''),
            'birth': x.get('birth', ''), 'death': x.get('death', ''), 'category': category,
            # the inventories count only the letters Gadda RECEIVED
            'to_gadda_inventories': 0, 'to_gadda_bonsanti': 0, 'to_gadda_liberati': 0, 'to_gadda_range': '',
            'described_sent': 0, 'described_received': 0, 'first_year': '', 'last_year': '',
        }
    if category and not p['category']:
        p['category'] = category
        if category in ORG_CATEGORIES:
            p['type'] = 'organization'
    return p


def new_tie(p, category):
    return {'tie': f't{p["id"][1:]}', 'correspondent_id': p['id'], 'correspondent': p['name'],
            'category': category, 'total': 0, 'bonsanti': 0, 'liberati': 0,
            'range': [], 'range_bonsanti': [], 'range_liberati': [], 'rows': 0}


actor(GADDA)
ties = {}
for r in pd.read_excel(SENDERS_XLSX).to_dict('records'):
    name = clean(r['Mittenti'])
    if not name:
        continue
    p = actor(name, clean(r['Tipologia']))
    t = ties.setdefault(p['id'], new_tie(p, clean(r['Tipologia'])))
    t['rows'] += 1
    for f, col in (('total', 'Numero dei pezzi'), ('bonsanti', 'Archivio Bonsanti'), ('liberati', 'Archivio Liberati')):
        if not pd.isna(r[col]):
            t[f] += int(r[col])
    for f, col in (('range', 'Estremi cronologici'), ('range_bonsanti', 'Estremi cronologici Archivio Bonsanti'),
                   ('range_liberati', 'Estremi cronologici Archivio Liberati')):
        if clean(r[col]):
            t[f].append(clean(r[col]))

print(len(people), 'actors,', len(ties), 'correspondences from the senders file')

1144 actors, 1143 correspondences from the senders file


## 7. Sources: where each letter is documented

A **published** letter's source is its **edition** (its sigla, e.g. `SIPELZDMS`, labelled with the
full reference); an **unpublished** one's is the **archive** holding it (Bonsanti or Liberati).
Editions link to their record on the site when there's a primary-bibliography record for them.

In [8]:
edition_record = collections.defaultdict(list)
for it in json.load(open(ITEMS_JSON, encoding='utf-8')):
    if it.get('corpus_lettere') and it.get('category') == 'primary bibliography':
        edition_record[it['corpus_lettere']].append(it['id'])
ARCHIVES = {
    'archivio a. bonsanti': ('archivio-bonsanti', 'Archivio A. Bonsanti'),
    'archivio bonsanti': ('archivio-bonsanti', 'Archivio A. Bonsanti'),
    'archivio liberati': ('archivio-liberati', 'Archivio Liberati'),
}
sources = {}


def edition_source(sigla, reference):
    if not sigla:
        return ''
    s = sources.setdefault(sigla, {'id': sigla, 'type': 'edition', 'sigla': sigla, 'label': reference,
                                   'opengadda_record': ' '.join(SITE_RECORD + i for i in sorted(edition_record.get(sigla, []))),
                                   'letters': 0})
    if reference and not s['label']:
        s['label'] = reference
    return s['id']


def archive_source(name):
    sid, label = ARCHIVES.get(key(name), (re.sub(r'\W+', '-', key(name)).strip('-'), clean(name)))
    sources.setdefault(sid, {'id': sid, 'type': 'archival collection', 'sigla': '', 'label': label,
                             'opengadda_record': '', 'letters': 0})
    return sid

## 8. The letters

For each row of `lettere.xlsx`:
- **date** → a start and an end (the same day when the date is exact); uncertain dates such as
  "16/12/1942 (?)" keep their original wording; missing dates stay empty;
- **sender(s)** and **recipient(s)** → actors (several names are separated by `;`). A letter with
  **several recipients** becomes one row per recipient (`Let123-1`, `Let123-2`), all pointing to the
  same `document`. Several senders don't need this: the model allows a letter to have several authors;
- **places** → their place ids (see step 5);
- **source** → its edition or archive (step 7);
- **tie** → the correspondence it belongs to: the person other than Gadda. People who appear only in
  the letters (mostly people Gadda wrote *to*) get a tie here, since they're not in the senders file.

Anything missing is recorded in `problems`, for the report.

In [9]:
def date_of(letter_id, v):
    "→ (start, end, original text, precision)"
    if letter_id in DATE_FIX:
        d = DATE_FIX[letter_id]
        return (d, d, clean(v), 'day') if d else ('', '', clean(v), 'unknown')
    if hasattr(v, 'strftime'):                   # a real date cell
        d = v.strftime('%Y-%m-%d')
        return d, d, '', 'day'
    text = clean(v)
    m = re.match(r'(\d{1,2})/(\d{1,2})/(\d{4})', text)
    if m:                                        # e.g. "16/12/1942 (?)": that day, uncertain
        d = f'{m[3]}-{int(m[2]):02d}-{int(m[1]):02d}'
        return d, d, text, 'day-uncertain'
    return '', '', text, 'unknown'


def letter_viaf(name, letter_id):
    "the VIAF reconciled in OpenRefine for this very occurrence, if any"
    occ = by_item.get((letter_id, key(name)))
    return occ[VIAF_COL] if occ else ''


letters, problems = [], collections.defaultdict(list)
for r in pd.read_excel(LETTERS_XLSX).to_dict('records'):
    lid = clean(r['Id'])
    senders = split_names(r['Mittente'])
    recipients = split_names(r['Destinatario']) or ['']
    start, end, date_text, precision = date_of(lid, r['Data'])
    pfrom, pto = place_of(r['Luogo di invio']), place_of(r['Luogo destinatario'])
    year = start[:4]
    sender_actors = [actor(s) for s in senders]
    for p in sender_actors:                      # counted once per document, even when split
        p['described_sent'] += 1
    for i, rcp in enumerate(recipients, 1):
        rid = lid if len(recipients) == 1 else f'{lid}-{i}'
        ra = actor(rcp) if rcp else None
        if ra:
            ra['described_received'] += 1
        others = [p for p in sender_actors + ([ra] if ra else []) if p['name'] != GADDA]
        for p in others:
            if year:
                p['first_year'] = min(filter(None, [p['first_year'], year]))
                p['last_year'] = max(filter(None, [p['last_year'], year]))
            ties.setdefault(p['id'], new_tie(p, clean(r['Genere'])))
        letters.append({
            'letter': rid, 'document': lid,
            'sender_ids': '; '.join(p['id'] for p in sender_actors), 'senders': '; '.join(senders),
            'recipient_id': ra['id'] if ra else '', 'recipient': rcp,
            'date_start': start, 'date_end': end, 'date_text': date_text, 'date_precision': precision,
            'from_place_id': pfrom['id'], 'from_raw': pfrom['raw'], 'from_city': pfrom['city'],
            'from_address': pfrom['address'], 'from_geonames': pfrom['geonames'],
            'to_place_id': pto['id'], 'to_raw': pto['raw'], 'to_city': pto['city'],
            'to_address': pto['address'], 'to_geonames': pto['geonames'],
            'edition_sigla': clean(r['Opera']), 'edition': clean(r['Sigla sciolta']),
            'archive': clean(r['Archivio']),
            'source_id': edition_source(clean(r['Opera']), clean(r['Sigla sciolta'])) or
                         (archive_source(r['Archivio']) if clean(r['Archivio']) else ''),
            'tie_ids': '; '.join(ties[p['id']]['tie'] for p in others),
            'category': clean(r['Genere']),
        })
        if not letters[-1]['source_id']:
            problems['letter without source'].append(lid)
        if not senders:
            problems['letter without sender'].append(lid)
        if not rcp:
            problems['letter without recipient'].append(lid)
        if precision == 'unknown':
            problems['letter without date'].append(lid)
        for p in (pfrom, pto):
            if p['raw'] and not p['geonames']:
                problems[f'place without GeoNames id: {p["raw"]}'].append(lid)
        for n in senders + ([rcp] if rcp else []):
            v = letter_viaf(n, lid)
            if v and v not in people[key(n)]['viaf']:
                problems[f'VIAF of {n} differs in this letter ({v})'].append(lid)

print(len(letters), 'letter rows from', len({x['document'] for x in letters}), 'documents')
pd.DataFrame(letters).head(3)

2481 letter rows from 2473 documents


,letter,document,sender_ids,senders,recipient_id,recipient,date_start,date_end,date_text,date_precision,...,to_raw,to_city,to_address,to_geonames,edition_sigla,edition,archive,source_id,tie_ids,category
0,Let1,Let1,p0146,Bonsanti Alessandro,p0001,Gadda Carlo Emilio,1930-06-22,1930-06-22,,day,...,,,,,SIPELZDMS,"C.E. Gadda-A. Bonsanti, Sono il pero e la zucc...",,SIPELZDMS,t0146,Contatti letterari
1,Let2,Let2,p0146,Bonsanti Alessandro,p0001,Gadda Carlo Emilio,1930-07-30,1930-07-30,,day,...,Rheinland,Rhineland-Palatinate,,2847618,SIPELZDMS,"C.E. Gadda-A. Bonsanti, Sono il pero e la zucc...",,SIPELZDMS,t0146,Contatti letterari
2,Let3,Let3,p0146,Bonsanti Alessandro,p0001,Gadda Carlo Emilio,1931-01-12,1931-01-12,,day,...,Milano (Simpliciano 2),Milan,Simpliciano 2,3173435,SIPELZDMS,"C.E. Gadda-A. Bonsanti, Sono il pero e la zucc...",,SIPELZDMS,t0146,Contatti letterari


## 9. Putting it together: sources and statistics

- A **correspondence's** sources are the inventories that count it (Bonsanti and/or Liberati) plus
  the editions/archives of its individual letters; an **actor's** sources are those of their
  correspondences.
- Each actor gets their **statistics**: letters to Gadda counted in the inventories (overall, per
  archive, date range), and the individually described letters in each direction.
- **Possible duplicates**: undated letters with the same sender, recipient and edition.

In [10]:
for x in letters:                                # letters per source (each document once)
    if x['source_id'] and (x['letter'] == x['document'] or x['letter'].endswith('-1')):
        sources[x['source_id']]['letters'] += 1

tie_sources = collections.defaultdict(set)
for t in ties.values():
    if t['bonsanti']:
        tie_sources[t['tie']].add(archive_source('Archivio Bonsanti'))
    if t['liberati']:
        tie_sources[t['tie']].add(archive_source('Archivio Liberati'))
for x in letters:
    for tid in filter(None, x['tie_ids'].split('; ')):
        if x['source_id']:
            tie_sources[tid].add(x['source_id'])
for t in ties.values():
    t['source_ids'] = '; '.join(sorted(tie_sources[t['tie']]))

actor_sources = collections.defaultdict(set)
for t in ties.values():
    actor_sources[t['correspondent_id']] |= tie_sources[t['tie']]
for x in letters:
    if x['source_id']:
        for pid in filter(None, x['sender_ids'].split('; ') + [x['recipient_id']]):
            actor_sources[pid].add(x['source_id'])
for p in people.values():
    p['source_ids'] = '; '.join(sorted(actor_sources[p['id']]))
    if not p['source_ids'] and key(p['name']) not in SET_ASIDE:
        problems['actor without source'].append(p['id'])

by_id = {p['id']: p for p in people.values()}
for t in ties.values():
    p = by_id[t['correspondent_id']]
    p['to_gadda_inventories'], p['to_gadda_bonsanti'], p['to_gadda_liberati'] = t['total'], t['bonsanti'], t['liberati']
    p['to_gadda_range'] = ' | '.join(t['range'])
    t['described_letters'] = p['described_sent'] + p['described_received']
    t['described_to_gadda'] = p['described_sent']
    t['described_from_gadda'] = p['described_received']

seen = collections.defaultdict(set)
for x in letters:
    if x['date_precision'] == 'unknown':
        seen[(x['senders'], x['recipient'], x['edition_sigla'])].add(x['document'])
dupes = [sorted(v) for v in seen.values() if len(v) > 1]

pd.DataFrame(people.values()).head(3)

,id,name,type,viaf,wikidata,birth,death,category,to_gadda_inventories,to_gadda_bonsanti,to_gadda_liberati,to_gadda_range,described_sent,described_received,first_year,last_year,source_ids
0,p0001,Gadda Carlo Emilio,person,56610080,Q378645,1893,1973,,0,0,0,,1517,955,,,AF; CAC; CEGFRVR; CNS; CaBo; CaCe; CaGia; CarR...
1,p0002,Accademia Nazionale dei Lincei,organization,159293044,,,,"Istituzioni, associazioni ed enti pubblici",7,7,6,1939-1969,0,0,,,archivio-bonsanti; archivio-liberati
2,p0003,Accademia Reale d'Italia,organization,,,,,"Istituzioni, associazioni ed enti pubblici",3,0,3,1942-1943,0,0,,,archivio-liberati


## 10. Writing the tables

Everything is saved as CSV (with a BOM, so Excel shows accents correctly). In `ties.csv` the
inventory columns are named `to_gadda_…`, because they count only the letters Gadda received.

In [11]:
def write(path, rows, fields):
    with open(path, 'w', newline='', encoding='utf-8-sig') as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
        w.writeheader()
        w.writerows(rows)


write(IDS_CSV, [{'kind': kd, 'key': k, 'id': v, 'label': labels.get((kd, k), '')}
                for (kd, k), v in sorted(ids.items(), key=lambda kv: (kv[0][0], kv[1]))],
      ['kind', 'key', 'id', 'label'])
write(HERE / 'people.csv', sorted(people.values(), key=lambda p: p['id']), list(next(iter(people.values()))))
write(HERE / 'ties.csv', [{'tie': t['tie'], 'correspondent_id': t['correspondent_id'], 'correspondent': t['correspondent'],
                           'category': t['category'], 'to_gadda_inventories': t['total'],
                           'to_gadda_bonsanti': t['bonsanti'], 'to_gadda_liberati': t['liberati'],
                           'to_gadda_range': ' | '.join(t['range']), 'to_gadda_range_bonsanti': ' | '.join(t['range_bonsanti']),
                           'to_gadda_range_liberati': ' | '.join(t['range_liberati']), 'rows': t['rows'],
                           'described_to_gadda': t['described_to_gadda'], 'described_from_gadda': t['described_from_gadda'],
                           'source_ids': t['source_ids']} for t in sorted(ties.values(), key=lambda t: t['tie'])],
      ['tie', 'correspondent_id', 'correspondent', 'category', 'to_gadda_inventories', 'to_gadda_bonsanti',
       'to_gadda_liberati', 'to_gadda_range', 'to_gadda_range_bonsanti', 'to_gadda_range_liberati', 'rows',
       'described_to_gadda', 'described_from_gadda', 'source_ids'])
write(HERE / 'sources.csv', sorted(sources.values(), key=lambda s: (s['type'], s['id'])),
      ['id', 'type', 'sigla', 'label', 'opengadda_record', 'letters'])
write(HERE / 'letters.csv', letters, list(letters[0]))

places = {}
for x in letters:
    for side in ('from', 'to'):
        if x[f'{side}_raw']:
            pl = places.setdefault(x[f'{side}_raw'], {'id': x[f'{side}_place_id'], 'raw': x[f'{side}_raw'],
                                                       'city': x[f'{side}_city'], 'address': x[f'{side}_address'],
                                                       'geonames': x[f'{side}_geonames'], 'letters': 0})
            pl['letters'] += 1
write(HERE / 'places.csv', sorted(places.values(), key=lambda p: -p['letters']),
      ['id', 'raw', 'city', 'address', 'geonames', 'letters'])
print('tables written to', HERE)

tables written to /Users/martinapensalfini/Documents/opengadda/lod/letters


## 11. The report

`report.md` lists what's missing and what needs a decision. Missing information stays missing in
the RDF (the property is simply left out). The list of people without a row in the senders file
only shows people who **wrote to** Gadda: people he only wrote to are rightly absent from it,
because it counts letters he received.

In [12]:
P = list(people.values())
in_senders = {t['correspondent_id'] for t in ties.values() if t['rows']}
not_in_senders = [p for p in P if p['name'] != GADDA and p['id'] not in in_senders
                  and p['described_sent'] and key(p['name']) not in SET_ASIDE]
docs = {x['document']: x for x in letters}
lines = ["# Gadda's correspondence: what needs checking", '',
         f'- actors: {len(P)} ({sum(p["type"] == "organization" for p in P)} organisations); '
         f'with VIAF {sum(1 for p in P if p["viaf"])}, with Wikidata {sum(1 for p in P if p["wikidata"])}',
         f'- correspondences (ties): {len(ties)}, of which {sum(1 for t in ties.values() if t["described_letters"])} '
         f'with individual letters; merged from two rows: ' + ', '.join(t['correspondent'] for t in ties.values() if t['rows'] > 1),
         f'- letters: {len(letters)} rows from {len(docs)} documents',
         f'- places: {len(places)} as written, {sum(1 for p in places.values() if p["geonames"])} with GeoNames',
         f'- VIAF list checked by the user: {"yes" if VIAF_CHECKED else "not yet"}', '']
lines += ['## Missing information (left out of the RDF, never invented)', '',
          f'- letters without a date: {sum(1 for x in docs.values() if x["date_precision"] == "unknown")}',
          f'- letters without a recipient: {sum(1 for x in letters if not x["recipient"])}',
          f'- letters without a sender: {sum(1 for x in letters if not x["senders"])}',
          f'- place mentions without GeoNames (kept as named places, no coordinates): '
          f'{sum(1 for x in letters for sd in ("from", "to") if x[sd + "_raw"] and not x[sd + "_geonames"])}',
          f'- actors with neither VIAF nor Wikidata (OpenGadda id only): {sum(1 for p in P if not p["viaf"] and not p["wikidata"])} of {len(P)}',
          f'- actors with VIAF but no Wikidata: {sum(1 for p in P if p["viaf"] and not p["wikidata"])}', '']
lines += ['## Actors with two VIAF ids (choose one)', '']
lines += [f'- {p["name"]} ({p["id"]}): {p["viaf"]}' for p in P if '|' in p['viaf']] or ['- none']
lines += ['', '## Possible duplicate letters (undated, same sender, recipient and edition)', '']
lines += [f'- {", ".join(v)}' for v in dupes] or ['- none']
lines += ['', '## People who wrote to Gadda but have no row in dati_mittenti_Gadda.xlsx (spelling difference, or missing from the inventories)', '']
lines += [f'- {p["name"]} ({p["id"]}, {p["described_sent"]} letters to Gadda)'
          for p in sorted(not_in_senders, key=lambda p: p['name'])] or ['- none']
for what, lids in sorted(problems.items()):
    lids = sorted(set(lids), key=lambda s: int(re.sub(r'\D', '', s) or 0))
    lines += ['', f'## {what}: {len(lids)}', '', '- ' + ', '.join(lids)]
(HERE / 'report.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('\n'.join(lines[:16]))

# Gadda's correspondence: what needs checking

- actors: 1163 (135 organisations); with VIAF 295, with Wikidata 128
- correspondences (ties): 1162, of which 510 with individual letters; merged from two rows: Gadda Enrico, Martini Quinto, Montano Lorenzo, Ronchetti Marisa, Ronchetti Tina
- letters: 2481 rows from 2473 documents
- places: 219 as written, 190 with GeoNames
- VIAF list checked by the user: yes

## Missing information (left out of the RDF, never invented)

- letters without a date: 10
- letters without a recipient: 3
- letters without a sender: 2
- place mentions without GeoNames (kept as named places, no coordinates): 60
- actors with neither VIAF nor Wikidata (OpenGadda id only): 863 of 1163
- actors with VIAF but no Wikidata: 172
